In [1]:
import torch
from torch import nn
import numpy as np
from torch.utils.data import DataLoader
from torch.optim import AdamW

In [2]:
from datasets import load_dataset
from transformers import ( AutoTokenizer ,
                          AutoModelForSequenceClassification ,
                          get_linear_schedule_with_warmup)

from sklearn.metrics import accuracy_score,classification_report
from tqdm import tqdm

In [1]:
!pip uninstall torchvision -y

In [ ]:
import os
os.kill(os.getpid(), 9)  # force restart runtime

In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)
print(torch.cuda.get_device_properties())
print(torch.cuda.get_device_name())
print(torch.cuda.get_device_properties().total_memory)

cuda
_CudaDeviceProperties(name='Tesla T4', major=7, minor=5, total_memory=14912MB, multi_processor_count=40, uuid=052711d4-8736-df0b-98da-b3e7b49f7c27, pci_bus_id=0, pci_device_id=4, pci_domain_id=0, L2_cache_size=4MB)
Tesla T4
15637086208


In [4]:
pip install torch transformers datasets scikit-learn tqdm accelerate

`Block 1 | Configuration`

In [5]:
CONFIG = {
    #model
    "model_name" : "bert-base-uncased",
    "num_labels" :2,


    #Tokenizer
    "max_length" :256,
    "truncation" :True,
    "padding"    :"max_length",

    # Training

    "batch_size"  :16,
    "num_epochs"  :3,
    "lr"          :2e-5,
    "weight_decay":0.01,
    "warmup_ratio":0.1,

    # Misc

    "seed"        :42,
    "save_path"   :"./bert_imdb_finetuned",
    "train_subset":None,
    "test_subset" :None,
}

`Block 2 | REPRODUCIBILITY & DEVICE SETUP`

In [6]:
torch.manual_seed(CONFIG["seed"])
np.random.seed(CONFIG["seed"])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


`BLOCK 3 │ LOAD RAW DATASET FROM HUGGINGFACE HUB`

In [7]:
raw_datasets = load_dataset("stanfordnlp/imdb")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [8]:
train_data = raw_datasets["train"]
print(len(train_data))
test_data = raw_datasets["test"]
print(len(test_data))


if CONFIG["train_subset"]:
  train_data = train_data.select(range(CONFIG["train_subset"]))

if CONFIG["test_subset"]:
    test_data  = test_data.select(range(CONFIG["test_subset"]))

print(f"[INFO] Train size: {len(train_data)} | Test size: {len(test_data)}")


25000
25000
[INFO] Train size: 25000 | Test size: 25000


`BLOCK 4 │ TOKENIZER + TOKENIZATION`

In [9]:
tokenizer = AutoTokenizer.from_pretrained(CONFIG["model_name"])


In [10]:
def tokenize_batch(batch):
    """
    Applied to each batch of examples by datasets.map().
    Returns a dict with keys:
      input_ids      – token indices for each token in the sequence
      attention_mask – 1 for real tokens, 0 for [PAD] tokens
      token_type_ids – segment ids (all 0 for single-sentence tasks)
      label          – kept as-is (0 or 1)
    """


    return tokenizer(
        batch["text"],
        max_length = CONFIG["max_length"],
        truncation = CONFIG["truncation"],
        padding    = CONFIG["padding"]
    )


In [11]:
# Map tokenization across all examples in parallel (batched=True is fast)

train_data = train_data.map(tokenize_batch, batched=True)
test_data  = test_data.map(tokenize_batch,  batched=True)



Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

In [12]:
# Tell HuggingFace datasets which columns are tensors and which are the label
train_data = train_data.rename_column("label", "labels")
test_data  = test_data.rename_column("label", "labels")


In [13]:
print(len(train_data[0]))
for key in train_data[0].keys():
    print(key)

print(train_data[0]['input_ids'])

5
text
labels
input_ids
token_type_ids
attention_mask
[101, 1045, 12524, 1045, 2572, 8025, 1011, 3756, 2013, 2026, 2678, 3573, 2138, 1997, 2035, 1996, 6704, 2008, 5129, 2009, 2043, 2009, 2001, 2034, 2207, 1999, 3476, 1012, 1045, 2036, 2657, 2008, 2012, 2034, 2009, 2001, 8243, 2011, 1057, 1012, 1055, 1012, 8205, 2065, 2009, 2412, 2699, 2000, 4607, 2023, 2406, 1010, 3568, 2108, 1037, 5470, 1997, 3152, 2641, 1000, 6801, 1000, 1045, 2428, 2018, 2000, 2156, 2023, 2005, 2870, 1012, 1026, 7987, 1013, 1028, 1026, 7987, 1013, 1028, 1996, 5436, 2003, 8857, 2105, 1037, 2402, 4467, 3689, 3076, 2315, 14229, 2040, 4122, 2000, 4553, 2673, 2016, 2064, 2055, 2166, 1012, 1999, 3327, 2016, 4122, 2000, 3579, 2014, 3086, 2015, 2000, 2437, 2070, 4066, 1997, 4516, 2006, 2054, 1996, 2779, 25430, 14728, 2245, 2055, 3056, 2576, 3314, 2107, 2004, 1996, 5148, 2162, 1998, 2679, 3314, 1999, 1996, 2142, 2163, 1012, 1999, 2090, 4851, 8801, 1998, 6623, 7939, 4697, 3619, 1997, 8947, 2055, 2037, 10740, 2006, 4331, 1010,

In [14]:
#Remove raw text column
train_data.set_format(type="torch", columns=["input_ids", "attention_mask","token_type_ids", "labels"])
test_data.set_format(type="torch", columns=["input_ids", "attention_mask","token_type_ids", "labels"])

`DataLoaders`

In [15]:
train_loader = DataLoader(train_data, batch_size=CONFIG["batch_size"], shuffle=True)
test_loader  = DataLoader(test_data, batch_size=CONFIG["batch_size"], shuffle=False)
print(f"{len(train_loader)} | {len(test_loader)}")

1563 | 1563


`MODEL`
#####  AutoModelForSequenceClassification loads BERT + a linear classification head

##### on top of the [CLS] token representation.
##### The head is randomly initialized; all other weights come from the pretrained checkpoint.



In [16]:
model = AutoModelForSequenceClassification.from_pretrained(
    CONFIG["model_name"],
    num_labels = CONFIG["num_labels"]
)
model.to(device)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [17]:
# Parameter Count

total_params = sum(p.numel() for p in model.parameters())
print(f"Total Parameters: {total_params}")
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"[INFO] Total params: {total_params:,} | Trainable: {trainable_params:,}")


Total Parameters: 109483778
[INFO] Total params: 109,483,778 | Trainable: 109,483,778


`BLOCK 7 │ OPTIMIZER & LR SCHEDULER`

In [ ]:
"""Bias terms — they're just scalar offsets like y = wx + b.
   Shrinking them toward zero adds no regularization benefit
   and can hurt the model's ability to shift activations correctly.


   LayerNorm weights — LayerNorm has two parameters: weight (scale γ)
   and bias (shift β). Their whole job is to rescale and reshift the
   distribution of activations. If you decay them toward zero, you're
   fighting against what LayerNorm is supposed to do. """"

In [18]:
no_decay = ["bias", "LayerNorm.weight"]

optimizer_grouped_params = [
    {
        "params"      : [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)],
        "weight_decay": CONFIG["weight_decay"],
    },
    {
        "params"      : [p for n, p in model.named_parameters() if any(nd in n for nd in no_decay)],
        "weight_decay": 0.0,
    },
]

In [19]:
optimizer = AdamW(optimizer_grouped_params, lr=CONFIG["lr"])

# Linear warmup then linear decay — avoids catastrophic forgetting at the start
total_steps  = len(train_loader) * CONFIG["num_epochs"]
warmup_steps = int(total_steps * CONFIG["warmup_ratio"])


scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps   = warmup_steps,
    num_training_steps = total_steps,
)

print(f"Total training steps: {total_steps} | Warmup steps: {warmup_steps}")


Total training steps: 4689 | Warmup steps: 468


`BLOCK 8 | Training Loop`

In [20]:
def train_one_epoch(model, loader, optimizer, scheduler, device, epoch):
    """
    One full pass over the training set.
    Returns average cross-entropy loss for the epoch.
    """
    model.train()          # activates dropout, batch norm in training mode
    total_loss = 0.0

    progress = tqdm(loader, desc=f"Epoch {epoch+1} [Train]", leave=False)

    for step, batch in enumerate(progress):
        # ── Move batch tensors to GPU/CPU ──────────────────────
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        token_type_ids = batch["token_type_ids"].to(device)
        labels         = batch["labels"].to(device)

        # ── Zero gradients from the previous step ─────────────
        optimizer.zero_grad()

        # ── Forward pass ──────────────────────────────────────
        # The model returns a SequenceClassifierOutput object.
        # When labels are passed, it automatically computes cross-entropy loss.
        outputs = model(
            input_ids      = input_ids,
            attention_mask = attention_mask,
            token_type_ids = token_type_ids,
            labels         = labels,
        )

        loss   = outputs.loss    # scalar cross-entropy loss
        logits = outputs.logits  # shape: (batch_size, num_labels)

        # ── Backward pass ─────────────────────────────────────
        loss.backward()          # compute gradients

        # Gradient clipping: prevents exploding gradients (common for transformers)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        # ── Optimizer + scheduler step ────────────────────────
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        progress.set_postfix(loss=f"{loss.item():.4f}", lr=f"{scheduler.get_last_lr()[0]:.2e}")

    avg_loss = total_loss / len(loader)
    return avg_loss


` BLOCK 9 │ EVALUATION LOOP`

In [21]:
def evaluate(model, loader, device, split_name="Val"):
    """
    Runs inference over `loader` and returns loss, accuracy, and a
    full sklearn classification report (precision, recall, F1).
    """
    model.eval()          # disables dropout, uses running stats for BN
    total_loss   = 0.0
    all_preds    = []
    all_labels   = []

    with torch.no_grad():   # no gradient tracking needed during inference
        progress = tqdm(loader, desc=f"[{split_name}]", leave=False)

        for batch in progress:
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            token_type_ids = batch["token_type_ids"].to(device)
            labels         = batch["labels"].to(device)

            outputs = model(
                input_ids      = input_ids,
                attention_mask = attention_mask,
                token_type_ids = token_type_ids,
                labels         = labels,
            )

            total_loss += outputs.loss.item()

            # Convert logits → predicted class (argmax over label dimension)
            preds = torch.argmax(outputs.logits, dim=-1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader)
    accuracy = accuracy_score(all_labels, all_preds)
    report   = classification_report(all_labels, all_preds, target_names=["negative", "positive"])
    return avg_loss, accuracy, report

`BLOCK 10 │ MAIN TRAINING ENTRY POINT`

In [22]:
print("\n" + "="*60)
print("  Starting Fine-Tuning")
print("="*60)

best_accuracy = 0.0

for epoch in range(CONFIG["num_epochs"]):
    # ── Train ─────────────────────────────────────────────────
    train_loss = train_one_epoch(model, train_loader, optimizer, scheduler, device, epoch)

    # ── Evaluate on test set ──────────────────────────────────
    test_loss, test_acc, test_report = evaluate(model, test_loader, device, split_name="Test")

    print(f"\nEpoch {epoch+1}/{CONFIG['num_epochs']}")
    print(f"  Train Loss : {train_loss:.4f}")
    print(f"  Test  Loss : {test_loss:.4f}  |  Accuracy : {test_acc*100:.2f}%")
    print(f"\n{test_report}")

    # ── Save best model ───────────────────────────────────────
    if test_acc > best_accuracy:
        best_accuracy = test_acc
        model.save_pretrained(CONFIG["save_path"])
        tokenizer.save_pretrained(CONFIG["save_path"])
        print(f"  [SAVED] New best model → accuracy: {best_accuracy*100:.2f}%")

print("\n" + "="*60)
print(f"  Training complete.  Best Test Accuracy: {best_accuracy*100:.2f}%")
print(f"  Model saved to: {CONFIG['save_path']}")
print("="*60)


  Starting Fine-Tuning



Epoch 1/3
  Train Loss : 0.3182
  Test  Loss : 0.2410  |  Accuracy : 90.21%

              precision    recall  f1-score   support

    negative       0.96      0.84      0.90     12500
    positive       0.86      0.97      0.91     12500

    accuracy                           0.90     25000
   macro avg       0.91      0.90      0.90     25000
weighted avg       0.91      0.90      0.90     25000



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  [SAVED] New best model → accuracy: 90.21%



Epoch 2/3
  Train Loss : 0.1721
  Test  Loss : 0.2536  |  Accuracy : 91.98%

              precision    recall  f1-score   support

    negative       0.91      0.93      0.92     12500
    positive       0.93      0.91      0.92     12500

    accuracy                           0.92     25000
   macro avg       0.92      0.92      0.92     25000
weighted avg       0.92      0.92      0.92     25000



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  [SAVED] New best model → accuracy: 91.98%



Epoch 3/3
  Train Loss : 0.0886
  Test  Loss : 0.3512  |  Accuracy : 92.15%

              precision    recall  f1-score   support

    negative       0.92      0.92      0.92     12500
    positive       0.92      0.92      0.92     12500

    accuracy                           0.92     25000
   macro avg       0.92      0.92      0.92     25000
weighted avg       0.92      0.92      0.92     25000



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  [SAVED] New best model → accuracy: 92.15%

  Training complete.  Best Test Accuracy: 92.15%
  Model saved to: ./bert_imdb_finetuned


`BLOCK 11 │ INFERENCE ON NEW TEXT  (post-training usage)`

In [23]:
def predict(texts: list[str], model_path: str = CONFIG["save_path"]) -> list[dict]:
    """
    Load the saved model and run inference on a list of raw strings.

    Args:
        texts      : list of review strings
        model_path : path to the saved fine-tuned model directory

    Returns:
        list of dicts with keys: 'text', 'label', 'confidence'
    """
    # Load saved artifacts
    inf_tokenizer = AutoTokenizer.from_pretrained(model_path)
    inf_model     = AutoModelForSequenceClassification.from_pretrained(model_path)
    inf_model     = inf_model.to(device)
    inf_model.eval()

    id2label = {0: "negative", 1: "positive"}
    results  = []

    # Tokenize in one batch
    encoding = inf_tokenizer(
        texts,
        max_length  = CONFIG["max_length"],
        truncation  = CONFIG["truncation"],
        padding     = CONFIG["padding"],
        return_tensors = "pt",
    )

    with torch.no_grad():
        outputs = inf_model(
            input_ids      = encoding["input_ids"].to(device),
            attention_mask = encoding["attention_mask"].to(device),
        )
        probs  = torch.softmax(outputs.logits, dim=-1)   # convert logits → probabilities
        preds  = torch.argmax(probs, dim=-1).cpu().numpy()
        confs  = probs.max(dim=-1).values.cpu().numpy()

    for text, pred, conf in zip(texts, preds, confs):
        results.append({
            "text"      : text[:80] + "..." if len(text) > 80 else text,
            "label"     : id2label[pred],
            "confidence": round(float(conf), 4),
        })

    return results

In [ ]:
sentences = [
    "The customer support team resolved my issue quickly and professionally.",
    "I absolutely loved the movie; the acting and storyline were outstanding.",
    "This laptop performs exceptionally well and exceeded my expectations.",
    "The restaurant served delicious food and provided excellent service.",
    "I'm very happy with the quality of this product and would recommend it to others."
]

In [27]:
# ── Quick smoke test ──────────────────────────────────────────
sample_reviews = [
    "The customer support team resolved my issue quickly and professionally.",
    "I absolutely loved the movie; the acting and storyline were outstanding.",
    "This laptop performs exceptionally well and exceeded my expectations.",
    "The restaurant served delicious food and provided excellent service.",
    "I'm very happy with the quality of this product and would recommend it to others."
]

print("\n[INFERENCE DEMO]")
for res in predict(sample_reviews):
    print(f"  [{res['label'].upper():8s}  {res['confidence']:.2%}]  {res['text']}")



[INFERENCE DEMO]


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

  [POSITIVE  98.54%]  The customer support team resolved my issue quickly and professionally.
  [POSITIVE  99.88%]  I absolutely loved the movie; the acting and storyline were outstanding.
  [POSITIVE  99.78%]  This laptop performs exceptionally well and exceeded my expectations.
  [POSITIVE  99.74%]  The restaurant served delicious food and provided excellent service.
  [POSITIVE  99.56%]  I'm very happy with the quality of this product and would recommend it to others...


In [28]:
negative_sentences = [
    "The application keeps crashing, making it almost impossible to use.",
    "I was extremely disappointed by the poor quality of the product.",
    "The service was slow, and the staff seemed uninterested in helping customers.",
    "This was a complete waste of time and money.",
    "The package arrived damaged and several items were missing."
]


print("\n[INFERENCE DEMO]")
for res in predict(negative_sentences):
    print(f"  [{res['label'].upper():8s}  {res['confidence']:.2%}]  {res['text']}")


[INFERENCE DEMO]


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

  [NEGATIVE  98.40%]  The application keeps crashing, making it almost impossible to use.
  [NEGATIVE  99.82%]  I was extremely disappointed by the poor quality of the product.
  [NEGATIVE  99.57%]  The service was slow, and the staff seemed uninterested in helping customers.
  [NEGATIVE  99.84%]  This was a complete waste of time and money.
  [NEGATIVE  67.96%]  The package arrived damaged and several items were missing.
